In [3]:
#import
from torch import nn,optim
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import numpy as np
from matplotlib import pyplot as plt
import os
import copy
import math
import sys
import importlib
from tqdm.auto import tqdm

In [4]:
# 코랩 환경인지 확인하는 조건문
if 'google.colab' in sys.modules:
    print("현재 환경: Google Colab")
    # 코랩 전용 설정 (예: 드라이브 마운트)
    from google.colab import drive
    drive.mount('/content/drive')
    path='/content/drive/MyDrive/02_학업/02_연구 및 프로젝트/2512-2602_Dash 연구인턴/pytorch_practice'
    # path='/content/drive/MyDrive//pytorch_practice'
    sys.path.append(path)
else:
    print("현재 환경: Local Jupyter")
    # 로컬 전용 설정 (예: 경로 설정)
    # path = './'
    path = r'g:\내 드라이브\02_학업\02_연구 및 프로젝트\2512-2602_Dash 연구인턴\pytorch_practice'
# from my_module import *
import my_module3 as mm
print(f"작업 경로: {path}")

현재 환경: Google Colab
Mounted at /content/drive
작업 경로: /content/drive/MyDrive/02_학업/02_연구 및 프로젝트/2512-2602_Dash 연구인턴/pytorch_practice


In [5]:
importlib.reload(mm)
import my_module2 as mm

In [6]:
model_dir = os.path.join(path, 'download')
print(os.listdir(model_dir))

root=os.path.join(path, 'data','test')
os.makedirs(root, exist_ok=True)
print(os.listdir(root))

checkpoint_dir = os.path.join(path, 'checkpoints')
print(os.listdir(checkpoint_dir))

output_dir = './saved_results'
output_dir=os.path.join(checkpoint_dir,output_dir)
print(os.listdir(output_dir))

DEVICE= 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'current device: {DEVICE}')

['cifar10_vgg16_bn-6ee7ea24.pt']
['cifar-10-batches-py', 'cifar-10-python.tar.gz']
['ckpt_ep1.pt', 'ckpt_ep2.pt', 'ckpt_ep3.pt', 'ckpt_ep5.pt', 'ckpt_ep10.pt', 'ckpt_ep15.pt', 'ckpt_ep20.pt', 'CNN_CIFAR10_final_weights.pth', 'best_model.pt', 'saved_results']
['psa_results.pkl', 'psa_results.json', 'psa_re_results_torch.pt', 'psa_nm_re_results_torch.pt', 'psa_nm(34)_re_results_torch.pt', 'pruning_real_state.pth', 'psa_results_torch.pt', 'psa_nm_results_torch.pt', 'psa_vector_re_results_torch.pt', 'psa_kernel_re_results_torch.pt', 'psa_channel_re_results_torch.pt', 'psa_scaling_re_results_torch.pt']
current device: cuda


In [7]:
model = torch.hub.load("chenyaofo/pytorch-cifar-models", "cifar10_vgg16_bn", pretrained=True).to(DEVICE)

/usr/local/lib/python3.12/dist-packages/torch/hub.py:335: UserWarning: You are about to download and run code from an untrusted repository. In a future release, this won't be allowed. To add the repository to your trusted list, change the command to {calling_fn}(..., trust_repo=False) and a command prompt will appear asking for an explicit confirmation of trust, or load(..., trust_repo=True), which will assume that the prompt is to be answered with 'yes'. You can also use load(..., trust_repo='check') which will only prompt for confirmation if the repo is not already trusted. This will eventually be the default behaviour
  warnings.warn(


Downloading: "https://github.com/chenyaofo/pytorch-cifar-models/zipball/master" to /root/.cache/torch/hub/master.zip
Downloading: "https://github.com/chenyaofo/pytorch-cifar-models/releases/download/vgg/cifar10_vgg16_bn-6ee7ea24.pt" to /root/.cache/torch/hub/checkpoints/cifar10_vgg16_bn-6ee7ea24.pt


100%|██████████| 58.3M/58.3M [00:01<00:00, 41.4MB/s]


In [8]:
BATCH_SIZE=128
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

full_train_DS = datasets.CIFAR10(root=root, train=True, download=True, transform=transform_train)
test_DS = datasets.CIFAR10(root=root, train=False, download=True, transform=transform_test)

train_size = 45000
val_size = 5000
train_DS, val_DS = random_split(full_train_DS, [train_size, val_size])

train_DL=torch.utils.data.DataLoader(train_DS, batch_size=BATCH_SIZE, shuffle=True)
val_DL = torch.utils.data.DataLoader(val_DS, batch_size=BATCH_SIZE, shuffle=False)
test_DL=torch.utils.data.DataLoader(test_DS, batch_size=BATCH_SIZE, shuffle=False)

print(f"Data loaded: Train({len(train_DS)}), Val({len(val_DS)}), Test({len(test_DS)})")

Data loaded: Train(45000), Val(5000), Test(10000)


In [9]:
rcorrect,acc=mm.Test(model, test_DL, DEVICE)
print(f"Test accuracy: {rcorrect}/{len(test_DL.dataset)} ({acc} %)")

Test accuracy: 9416/10000 (94.2 %)


In [ ]:
input_sub=copy.deepcopy(torch.nn.Sequential(*list(model.features.children()),
                                            torch.nn.Flatten(start_dim=1),
                                            *list(model.classifier.children())[:3])).to(DEVICE)
# output_sub=torch.nn.Sequential(*list(model.classifier.children())).to(DEVICE)
# output_sub=copy.deepcopy(torch.nn.Sequential(*[module for module in model.classifier.children()
#       if not isinstance(module, torch.nn.Dropout)])).to(DEVICE)

output_sub=copy.deepcopy(list(model.classifier.children())[3]).to(DEVICE)
final_sub=copy.deepcopy(torch.nn.Sequential(*list(model.classifier.children())[4:])).to(DEVICE)

In [ ]:
print(input_sub.children)
print(output_sub)
print(final_sub)

<bound method Module.children of Sequential(
  (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU(inplace=True)
  (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (5): ReLU(inplace=True)
  (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (7): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (8): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (9): ReLU(inplace=True)
  (10): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (11): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (12): ReLU(inplace=True)
  (13): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (14): Conv2d(128, 256, kernel_size=(3

In [ ]:
print(model.children)

<bound method Module.children of VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU(inplace=True)
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (7): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (9): ReLU(inplace=True)
    (10): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (12): ReLU(inplace=True)
    (13): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode

In [ ]:
images, labels = next(iter(val_DL))
# eg_input = images[0:1].to(DEVICE)
# print(eg_input.shape)
input=input_sub(images[0:1].to(DEVICE))
output=output_sub(input)

In [ ]:
print(input.shape)
print(output.shape)
print(labels.shape)

# output=torch.ones_like(output)
pred = output.argmax(dim=1)
# pred=torch.ones(output.shape[0])*3
print(pred.shape)

# print(torch.sum(pred==labels).item())
# print(torch.sum(labels==3).item())

torch.Size([1, 512])
torch.Size([1, 512])
torch.Size([128])
torch.Size([1])


In [ ]:
# # print(len(list((output_sub.named_parameters()))))
# for p in output_sub.named_parameters():
#     # print(len(p))
#     print(p[0])
#     # print(p[1])

print(output_sub.get_submodule)


<bound method Module.get_submodule of Sequential(
  (0): Linear(in_features=512, out_features=512, bias=True)
  (1): ReLU(inplace=True)
  (2): Linear(in_features=512, out_features=512, bias=True)
  (3): ReLU(inplace=True)
  (4): Linear(in_features=512, out_features=10, bias=True)
)>


In [ ]:
bit_width = list(range(2, 20))
q_min = [ -2**(b-1) for b in bit_width ]
q_max = [  2**(b-1) - 1   for b in bit_width ]

In [ ]:
bit_num=8
idx=bit_num-2
q_max_=q_max[idx]
q_min_=q_min[idx]

S_X_list=[]
S_Y_list=[]
Z_X_list=[]
Z_Y_list=[]

# X_max=0
# Y_max=0

for images,_ in val_DL:
    input=input_sub(images.to(DEVICE))
    output=output_sub(input)

    r_x_max=input.amax(dim=1)
    r_x_min=input.amin(dim=1)
    # X_max=max(X_max, torch.max(r_x_max).item())

    S_x= (r_x_max - r_x_min) / (q_max_ - q_min_)
    Z_x=torch.round(q_min_ - r_x_min/S_x)

    r_y_max=output.amax(dim=1)
    r_y_min=output.amin(dim=1)
    # Y_max=max(Y_max, torch.max(r_y_max).item())
    S_y= (r_y_max - r_y_min) / (q_max_ - q_min_)
    Z_y=torch.round(q_min_ - r_y_min/S_y)


    S_X_list.append(S_x.mean().item())
    S_Y_list.append(S_y.mean().item())
    Z_X_list.append(Z_x.mean().item())
    Z_Y_list.append(Z_y.mean().item())

S_x=np.mean(S_X_list)
Z_x=np.mean(np.round(Z_X_list))
S_y=np.mean(S_Y_list)
Z_y=np.mean(np.round(Z_Y_list))

bias=list(model.classifier.children())[3].bias
weight=list(model.classifier.children())[3].weight

r_w_max=torch.max(torch.abs(weight))
r_b_max=torch.max(torch.abs(bias))
S_w= (r_w_max) / (q_max_ +1)
Z_w=0
S_b= S_w*S_x
Z_b=0

weight_tensor=weight.data.clone()
bias_tensor=bias.data.clone()
quanted_weight=torch.zeros_like(weight_tensor)
quanted_bias=torch.zeros_like(bias_tensor)
weight_dequanted=torch.zeros_like(weight_tensor)
bias_dequanted=torch.zeros_like(bias_tensor)

quanted_weight=torch.round(weight_tensor/S_w+Z_w)
weight_dequanted=(quanted_weight-Z_w)*S_w
quanted_bias=torch.round(bias_tensor/S_b+Z_b)
bias_dequanted=(quanted_bias-Z_b)*S_b


q_bias=quanted_bias - Z_x*quanted_weight.sum(dim=1)

In [ ]:
bit_num=8
idx=bit_num-2
q_max_=q_max[idx]
q_min_=q_min[idx]


with torch.no_grad():
    r_x_max=torch.zeros(1).to(DEVICE)
    r_x_min=torch.zeros(1).to(DEVICE)
    r_y_max=torch.zeros(1).to(DEVICE)
    r_y_min=torch.zeros(1).to(DEVICE)
    for images,_ in val_DL:
        input=input_sub(images.to(DEVICE))
        output=output_sub(input)

        r_x_max=torch.max(r_x_max, torch.max(input))
        r_x_min=torch.min(r_x_min, torch.min(input))

        r_y_max=torch.max(r_y_max, torch.max(output))
        r_y_min=torch.min(r_y_min, torch.min(output))

    S_x= (r_x_max - r_x_min) / (q_max_ - q_min_)
    Z_x=torch.round(q_min_ - r_x_min/S_x)

    S_y= (r_y_max - r_y_min) / (q_max_ - q_min_)
    Z_y=torch.round(q_min_ - r_y_min/S_y)

    bias=list(model.classifier.children())[3].bias
    weight=list(model.classifier.children())[3].weight

    r_w_max=torch.max(torch.abs(weight))
    r_b_max=torch.max(torch.abs(bias))
    S_w= (r_w_max) / (q_max_ +1)
    Z_w=0
    S_b= S_w*S_x
    Z_b=0

    weight_tensor=weight.data.clone()
    bias_tensor=bias.data.clone()
    quanted_weight=torch.zeros_like(weight_tensor)
    quanted_bias=torch.zeros_like(bias_tensor)
    weight_dequanted=torch.zeros_like(weight_tensor)
    bias_dequanted=torch.zeros_like(bias_tensor)

    quanted_weight=torch.round(weight_tensor/S_w+Z_w)
    weight_dequanted=(quanted_weight-Z_w)*S_w
    quanted_bias=torch.round(bias_tensor/S_b+Z_b)
    bias_dequanted=(quanted_bias-Z_b)*S_b


    q_bias=quanted_bias - Z_x*quanted_weight.sum(dim=1)

In [ ]:
val_iter=iter(val_DL)

In [ ]:
images,labels=next(val_iter)
print(images.shape)
print(labels.shape)

torch.Size([128, 3, 32, 32])
torch.Size([128])


In [ ]:
input=input_sub(images.to(DEVICE))
output=output_sub(input)

print(input.shape)
print(output.shape)

input_quanted= torch.round(input/S_x + Z_x)
output_tmp=torch.nn.functional.linear(input_quanted, quanted_weight, q_bias)
output_tmp=output_tmp*(S_w*S_x)
output_quanted=torch.round(output_tmp/S_y) + Z_y
output_dequanted=(output_quanted - Z_y)*S_y

print(output_tmp.shape)
print(output_quanted.shape)

torch.Size([128, 512])
torch.Size([128, 512])
torch.Size([128, 512])
torch.Size([128, 512])


In [ ]:
print(output_quanted.max().item(), output_quanted.min().item())
print(output[0][:5])
print(output_tmp[0][:5])
print(output_dequanted[0][:5])

107.0 -123.0
tensor([0.2223, 0.1099, 0.1243, 0.1160, 0.2533], grad_fn=<SliceBackward0>)
tensor([0.2225, 0.1084, 0.1234, 0.1151, 0.2535], grad_fn=<SliceBackward0>)
tensor([0.2228, 0.1084, 0.1224, 0.1144, 0.2529], grad_fn=<SliceBackward0>)


In [ ]:
model.eval() # test mode로 전환
with torch.no_grad(): #model.eval과 같이 항상 해야 함
    rcorrect1 = 0
    rcorrect2 = 0
    # rcorrect = 0
    num=len(test_DL.dataset)
    cnt=0
    for x_batch, y_batch in test_DL:
        input=input_sub(x_batch.to(DEVICE))
        y_batch = y_batch.to(DEVICE)

        # # inference
        input_quanted= torch.round(input/S_x + Z_x)
        output_tmp=torch.nn.functional.linear(input_quanted, quanted_weight, q_bias)
        output_tmp=output_tmp*(S_w*S_x)
        output_quanted=torch.round(output_tmp/S_y) + Z_y
        output_dequanted=(output_quanted - Z_y)*S_y
        y_hat=final_sub(output_dequanted)

        # output=output_sub(input)

        # y_hat=final_sub(output)
        # print(input.shape, output.shape, y_hat.shape)
        # x_batch = x_batch.to(DEVICE)
        # y_batch = y_batch.to(DEVICE)

        # corrects accumulation
        pred = y_hat.argmax(dim=1)
        corrects_b = torch.sum(pred == y_batch).item() # torch.eq(pred, y_batch).sum().item()
        rcorrect += corrects_b
        cnt+=1
        # print(f"Batch {cnt}/{len(test_DL)} processed")

    accuracy_e = rcorrect/len(test_DL.dataset)*100
print(f"Test accuracy: {rcorrect}/{len(test_DL.dataset)} ({accuracy_e:.1f} %)")


Test accuracy: 9416/10000 (94.2 %)


In [10]:
input_sub=copy.deepcopy(torch.nn.Sequential(*list(model.features.children()),
                                            torch.nn.Flatten(start_dim=1)).to(DEVICE))
FC_sub1=copy.deepcopy(list(model.classifier.children())[0]).to(DEVICE)
FC_sub2=copy.deepcopy(list(model.classifier.children())[3]).to(DEVICE)
FC_sub3=copy.deepcopy(list(model.classifier.children())[6]).to(DEVICE)


In [ ]:
bit_num=8
idx=bit_num-2
q_max_=q_max[idx]
q_min_=q_min[idx]


with torch.no_grad():
    r_x_max=torch.zeros(1).to(DEVICE)
    r_x_min=torch.zeros(1).to(DEVICE)
    r_y_max=torch.zeros(1).to(DEVICE)
    r_y_min=torch.zeros(1).to(DEVICE)
    for images,_ in val_DL:
        input=input_sub(images.to(DEVICE))

        fc_output1=FC_sub1(input)
        fc_input2=nn.ReLU()(fc_output1)

        fc_output2=FC_sub2(fc_input2)
        fc_input3=nn.ReLU()(fc_output2)

        output=FC_sub3(fc_input3)

        r_x_max=torch.max(r_x_max, torch.max(input))
        r_x_min=torch.min(r_x_min, torch.min(input))

        r_y_max=torch.max(r_y_max, torch.max(output))
        r_y_min=torch.min(r_y_min, torch.min(output))

    S_x= (r_x_max - r_x_min) / (q_max_ - q_min_)
    Z_x=torch.round(q_min_ - r_x_min/S_x)

    S_y= (r_y_max - r_y_min) / (q_max_ - q_min_)
    Z_y=torch.round(q_min_ - r_y_min/S_y)

    bias=list(model.classifier.children())[3].bias
    weight=list(model.classifier.children())[3].weight

    r_w_max=torch.max(torch.abs(weight))
    r_b_max=torch.max(torch.abs(bias))
    S_w= (r_w_max) / (q_max_ +1)
    Z_w=0
    S_b= S_w*S_x
    Z_b=0

    weight_tensor=weight.data.clone()
    bias_tensor=bias.data.clone()
    quanted_weight=torch.zeros_like(weight_tensor)
    quanted_bias=torch.zeros_like(bias_tensor)
    weight_dequanted=torch.zeros_like(weight_tensor)
    bias_dequanted=torch.zeros_like(bias_tensor)

    quanted_weight=torch.round(weight_tensor/S_w+Z_w)
    weight_dequanted=(quanted_weight-Z_w)*S_w
    quanted_bias=torch.round(bias_tensor/S_b+Z_b)
    bias_dequanted=(quanted_bias-Z_b)*S_b


    q_bias=quanted_bias - Z_x*quanted_weight.sum(dim=1)

In [12]:
bit_width = list(range(2, 20))
q_min = [ -2**(b-1) for b in bit_width ]
q_max = [  2**(b-1) - 1   for b in bit_width ]

In [161]:
bit_num=3
idx=bit_num-2
q_max_=q_max[idx]
q_min_=q_min[idx]

# 설정을 위한 리스트 준비 (FC1, FC2, FC3)
num_layers = 3
sub_layers = [FC_sub1, FC_sub2, FC_sub3]

# 각 레이어별 통계를 저장할 구조체
r_in_min = [torch.tensor(float('inf')).to(DEVICE) for _ in range(num_layers)]
r_in_max = [torch.tensor(float('-inf')).to(DEVICE) for _ in range(num_layers)]
r_out_min = [torch.tensor(float('inf')).to(DEVICE) for _ in range(num_layers)]
r_out_max = [torch.tensor(float('-inf')).to(DEVICE) for _ in range(num_layers)]

with torch.no_grad():
    for images, _ in val_DL:
        curr_in = input_sub(images.to(DEVICE))

        # 순차적으로 레이어를 통과하며 통계 수집
        for i in range(num_layers):
            # Input 통계 업데이트
            r_in_min[i] = torch.min(r_in_min[i], torch.min(curr_in))
            r_in_max[i] = torch.max(r_in_max[i], torch.max(curr_in))

            # 레이어 통과 (ReLU 포함 여부에 따라 수정 가능)


            if i < num_layers - 1:
                curr_out = nn.ReLU()(sub_layers[i](curr_in))
            else:
                curr_out = sub_layers[i](curr_in)

            # Output 통계 업데이트
            r_out_min[i] = torch.min(r_out_min[i], torch.min(curr_out))
            r_out_max[i] = torch.max(r_out_max[i], torch.max(curr_out))

            # # 다음 레이어의 입력을 위해 ReLU 적용 (마지막 레이어 제외 설정 등 가능)
            # if i < num_layers - 1:
            #     curr_in = nn.ReLU()(curr_out)
            # else:
            #     curr_in = curr_out
            curr_in = curr_out

# 결과를 저장할 딕셔너리
layer_params = {}

for i in range(num_layers):
    layer = sub_layers[i]

    # 1. Activation (Input/Output) Scale & Zero Point
    S_in = (r_in_max[i] - r_in_min[i]) / (q_max_ - q_min_)
    Z_in = torch.round(q_min_ - r_in_min[i] / S_in)

    S_out = (r_out_max[i] - r_out_min[i]) / (q_max_ - q_min_)
    Z_out = torch.round(q_min_ - r_out_min[i] / S_out)

    # 2. Weight & Bias Scale (Symmetric Quantization 가정)
    w = layer.weight.data
    b = layer.bias.data

    r_w_max = torch.max(torch.abs(w))
    S_w = r_w_max / (q_max_) # Symmetric: q_max_가 127(int8)인 경우
    Z_w = 0

    # Bias Scale은 관례적으로 S_in * S_w를 사용
    S_b = S_in * S_w
    Z_b = 0

    # 3. Quantization (Integer 값 생성)
    q_w = torch.round(w / S_w + Z_w).clamp(-q_max_-1, q_max_)
    q_b = torch.round(b / S_b + Z_b)

    # NPU 연산용 최적화된 Bias (Z_in 보정 포함)
    # 실제 하드웨어 식: Y_q = (W_q * X_q + Bias_eff) * (S_in*S_w/S_out)
    q_bias_eff = q_b - Z_in * q_w.sum(dim=1)

    layer_params[f'FC{i+1}'] = {
        'S_in': S_in, 'Z_in': Z_in,
        'S_out': S_out, 'Z_out': Z_out,
        'S_w': S_w, 'Z_w': Z_w,
        'S_b': S_b, 'q_w': q_w, 'q_b': q_b,
        'q_bias_eff': q_bias_eff
    }

In [163]:
print(len(layer_params))
print(list(layer_params.keys()))
for layer_name, params in layer_params.items():
    print(f"\n--- {layer_name} Parameters ---")
    for param_name, value in params.items():
        if isinstance(value, torch.Tensor):
            if value.dim() >0:
                print(f"{param_name}: {value.shape}")
            else:
                print(f"{param_name}: {value.item()}")
        else:
            print(f"{param_name}: {value}")


3
['FC1', 'FC2', 'FC3']

--- FC1 Parameters ---
S_in: 0.5909859538078308
Z_in: -4.0
S_out: 0.22876746952533722
Z_out: -4.0
S_w: 0.02532247081398964
Z_w: 0
S_b: 0.014965225011110306
q_w: torch.Size([512, 512])
q_b: torch.Size([512])
q_bias_eff: torch.Size([512])
Scale_factor_exp: -4.0
Scale_factor: 0.0625

--- FC2 Parameters ---
S_in: 0.22876746952533722
Z_in: -4.0
S_out: 0.2455499768257141
Z_out: -4.0
S_w: 0.019548937678337097
Z_w: 0
S_b: 0.004472161177545786
q_w: torch.Size([512, 512])
q_b: torch.Size([512])
q_bias_eff: torch.Size([512])
Scale_factor_exp: -6.0
Scale_factor: 0.015625

--- FC3 Parameters ---
S_in: 0.2455499768257141
Z_in: -4.0
S_out: 3.4539318084716797
Z_out: -1.0
S_w: 0.05032747611403465
Z_w: 0
S_b: 0.01235791016370058
q_w: torch.Size([10, 512])
q_b: torch.Size([10])
q_bias_eff: torch.Size([10])
Scale_factor_exp: -8.0
Scale_factor: 0.00390625


In [162]:
for layer_name, params in layer_params.items():
    scale_factor=(params["S_w"]*params["S_in"])/params["S_out"]
    log2_scale = torch.log2(scale_factor)
    rounded_log2_scale = torch.round(log2_scale)
    params["Scale_factor_exp"] = rounded_log2_scale
    params["Scale_factor"] = 2 ** rounded_log2_scale


In [158]:
val_iter=iter(val_DL)

In [122]:
relu = nn.ReLU().to(DEVICE)

In [159]:
images,label=next(val_iter)
# print(image.shape)
# print(label.shape)
input_sub.eval()
with torch.no_grad():
    input=input_sub(images.to(DEVICE))


    input_quanted= torch.round(input/layer_params['FC1']['S_in'] + layer_params['FC1']['Z_in'])
    output1_tmp=torch.nn.functional.linear(input_quanted, layer_params['FC1']['q_w'], layer_params['FC1']['q_bias_eff'])
    input2_tmp=torch.nn.ReLU()(output1_tmp)

    input2_quanted=torch.round(input2_tmp*layer_params['FC1']['Scale_factor'])+layer_params['FC1']['Z_out']
    output2_tmp=torch.nn.functional.linear(input2_quanted, layer_params['FC2']['q_w'], layer_params['FC2']['q_bias_eff'])
    input3_tmp=torch.nn.ReLU()(output2_tmp)

    input3_quanted=torch.round(input3_tmp*layer_params['FC2']['Scale_factor'])+layer_params['FC2']['Z_out']
    output3_tmp=torch.nn.functional.linear(input3_quanted, layer_params['FC3']['q_w'], layer_params['FC3']['q_bias_eff'])
    final_quanted=torch.round(output3_tmp*layer_params['FC3']['Scale_factor'])+layer_params['FC3']['Z_out']

    # y_hat=(final_quanted-layer_params['FC3']['Z_out'])*layer_params['FC3']['S_out']
    y_hat=final_quanted
    # print(y_hat)

    y=FC_sub3(relu(FC_sub2(relu(FC_sub1(input)))))
    y2=FC_sub3(relu(FC_sub2(relu(FC_sub1(layer_params['FC1']['S_in']*(input_quanted-layer_params['FC1']['Z_in']))))))
    # print(y)

In [130]:
with torch.no_grad():
    torch.set_printoptions(precision=4,sci_mode=False)
    # print(relu(FC_sub1(input))[0][:10].detach().cpu())
    # print((input2_tmp*layer_params['FC1']['S_in']*layer_params['FC1']['S_w'])[0][:10].detach().cpu())
    print(FC_sub2(relu(FC_sub1(input)))[0][:10].detach().cpu())
    print((output2_tmp*layer_params['FC2']['S_in']*layer_params['FC2']['S_w'])[0][:10].detach().cpu())

tensor([ 0.7462,  0.0128, -0.0740,  0.8969,  0.0898,  0.9648,  0.3138, -0.0209,
         0.0130, -0.0000])
tensor([ 0.4770, -0.0044, -0.0703,  0.5384,  0.0472,  0.5651,  0.1895,  0.0071,
         0.0000,  0.0000])


In [160]:
print(y_hat[0])
print(y2[0])
print(y[0])

tensor([-1., -1., -1., -1., -1., -1., -1., -1., -1., -1.], device='cuda:0')
tensor([ 0.4993, -0.4543, -1.0438, -0.7453, -1.4398, -2.5955, -1.0811, -1.8576,
         8.5648,  0.1515], device='cuda:0')
tensor([ 0.1074, -0.2955, -1.1704, -1.1831, -1.9095, -2.7878, -1.4348, -2.1654,
        11.0623, -0.2257], device='cuda:0')


In [127]:
#뭔가 잘못된듯

input_sub.eval()
with torch.no_grad(): #model.eval과 같이 항상 해야 함
    rcorrect = 0
    num=len(test_DL.dataset)
    cnt=0
    for x_batch, y_batch in test_DL:
        input=input_sub(x_batch.to(DEVICE))
        y_batch = y_batch.to(DEVICE)

        # # inference
        input_quanted= torch.round(input/layer_params['FC1']['S_in'] + layer_params['FC1']['Z_in'])

        output1_quanted=torch.nn.functional.linear(input_quanted, layer_params['FC1']['q_w'], layer_params['FC1']['q_bias_eff'])
        input2_quanted=torch.nn.ReLU()(output1_quanted)
        input2_dequanted=(input2_quanted - layer_params['FC1']['Z_out'])*layer_params['FC1']['S_out']

        input2_quanted2=torch.round(input2_dequanted/layer_params['FC2']['S_in'] + layer_params['FC2']['Z_in'])
        output2_quanted=torch.nn.functional.linear(input2_quanted2, layer_params['FC2']['q_w'], layer_params['FC2']['q_bias_eff'])
        input3_quanted=torch.nn.ReLU()(output2_quanted)
        input3_dequanted=(input3_quanted - layer_params['FC2']['Z_out'])*layer_params['FC2']['S_out']

        input3_quanted2=torch.round(input3_dequanted/layer_params['FC2']['S_out'] + layer_params['FC2']['Z_out'])

        output3_quanted=torch.nn.functional.linear(input3_quanted2, layer_params['FC3']['q_w'], layer_params['FC3']['q_bias_eff'])
        output3_dequanted=(output3_quanted - layer_params['FC3']['Z_out'])*layer_params['FC3']['S_out']

        y_hat=output3_dequanted
        # output_tmp=torch.nn.functional.linear(input_quanted, quanted_weight, q_bias)
        # output_tmp=output_tmp*(S_w*S_x)
        # output_quanted=torch.round(output_tmp/S_y) + Z_y
        # output_dequanted=(output_quanted - Z_y)*S_y



        # corrects accumulation
        pred = y_hat.argmax(dim=1)
        corrects_b = torch.sum(pred == y_batch).item() # torch.eq(pred, y_batch).sum().item()
        rcorrect += corrects_b
        cnt+=1
        # print(f"Batch {cnt}/{len(test_DL)} processed")

    accuracy_e = rcorrect/len(test_DL.dataset)*100
print(f"Test accuracy: {rcorrect}/{len(test_DL.dataset)} ({accuracy_e:.1f} %)")


Test accuracy: 9407/10000 (94.1 %)


In [26]:
input_sub.eval() # test mode로 전환
with torch.no_grad(): #model.eval과 같이 항상 해야 함
    rcorrect = 0
    num=len(test_DL.dataset)
    cnt=0
    for x_batch, y_batch in test_DL:
        input=input_sub(x_batch.to(DEVICE))
        y_batch = y_batch.to(DEVICE)

        # # inference
        input_quanted= torch.round(input/layer_params['FC1']['S_in'] + layer_params['FC1']['Z_in'])

        output1_quanted=torch.nn.functional.linear(input_quanted, layer_params['FC1']['q_w'], layer_params['FC1']['q_bias_eff'])
        input2_quanted = torch.clamp(output1_quanted, min=layer_params['FC1']['Z_out'])
        input2_dequanted=(input2_quanted - layer_params['FC1']['Z_out'])*layer_params['FC1']['S_out']

        input2_quanted2=torch.round(input2_dequanted/layer_params['FC2']['S_in'] + layer_params['FC2']['Z_in'])
        output2_quanted=torch.nn.functional.linear(input2_quanted2, layer_params['FC2']['q_w'], layer_params['FC2']['q_bias_eff'])
        input3_quanted = torch.clamp(output2_quanted, min=layer_params['FC2']['Z_out'])
        input3_dequanted=(input3_quanted - layer_params['FC2']['Z_out'])*layer_params['FC2']['S_out']

        input3_quanted2=torch.round(input3_dequanted/layer_params['FC2']['S_out'] + layer_params['FC2']['Z_out'])

        output3_quanted=torch.nn.functional.linear(input3_quanted2, layer_params['FC3']['q_w'], layer_params['FC3']['q_bias_eff'])
        output3_dequanted=(output3_quanted - layer_params['FC3']['Z_out'])*layer_params['FC3']['S_out']

        y_hat=output3_dequanted
        # output_tmp=torch.nn.functional.linear(input_quanted, quanted_weight, q_bias)
        # output_tmp=output_tmp*(S_w*S_x)
        # output_quanted=torch.round(output_tmp/S_y) + Z_y
        # output_dequanted=(output_quanted - Z_y)*S_y



        # corrects accumulation
        pred = y_hat.argmax(dim=1)
        corrects_b = torch.sum(pred == y_batch).item() # torch.eq(pred, y_batch).sum().item()
        rcorrect += corrects_b
        cnt+=1
        # print(f"Batch {cnt}/{len(test_DL)} processed")

    accuracy_e = rcorrect/len(test_DL.dataset)*100
print(f"Test accuracy: {rcorrect}/{len(test_DL.dataset)} ({accuracy_e:.1f} %)")


Test accuracy: 9408/10000 (94.1 %)


In [ ]:
input_sub.eval() # test mode로 전환
with torch.no_grad(): #model.eval과 같이 항상 해야 함
    rcorrect = 0
    num=len(test_DL.dataset)
    cnt=0
    for x_batch, y_batch in test_DL:
        input=input_sub(x_batch.to(DEVICE))
        y_batch = y_batch.to(DEVICE)

        # # inference
        input_quanted= torch.round(input/layer_params['FC1']['S_in'] + layer_params['FC1']['Z_in'])

        output1_quanted=torch.nn.functional.linear(input_quanted, layer_params['FC1']['q_w'], layer_params['FC1']['q_bias_eff'])
        input2_quanted=torch.nn.ReLU()(output1_quanted)
        input2_dequanted=(input2_quanted - layer_params['FC1']['Z_out'])*layer_params['FC1']['S_out']

        input2_quanted2=torch.round(input2_dequanted/layer_params['FC2']['S_in'] + layer_params['FC2']['Z_in'])
        output2_quanted=torch.nn.functional.linear(input2_quanted2, layer_params['FC2']['q_w'], layer_params['FC2']['q_bias_eff'])
        input3_quanted=torch.nn.ReLU()(output2_quanted)
        input3_dequanted=(input3_quanted - layer_params['FC2']['Z_out'])*layer_params['FC2']['S_out']

        input3_quanted2=torch.round(input3_dequanted/layer_params['FC2']['S_out'] + layer_params['FC2']['Z_out'])

        output3_quanted=torch.nn.functional.linear(input3_quanted2, layer_params['FC3']['q_w'], layer_params['FC3']['q_bias_eff'])
        output3_dequanted=(output3_quanted - layer_params['FC3']['Z_out'])*layer_params['FC3']['S_out']

        y_hat=output3_dequanted
        # output_tmp=torch.nn.functional.linear(input_quanted, quanted_weight, q_bias)
        # output_tmp=output_tmp*(S_w*S_x)
        # output_quanted=torch.round(output_tmp/S_y) + Z_y
        # output_dequanted=(output_quanted - Z_y)*S_y



        # corrects accumulation
        pred = y_hat.argmax(dim=1)
        corrects_b = torch.sum(pred == y_batch).item() # torch.eq(pred, y_batch).sum().item()
        rcorrect += corrects_b
        cnt+=1
        # print(f"Batch {cnt}/{len(test_DL)} processed")

    accuracy_e = rcorrect/len(test_DL.dataset)*100
print(f"Test accuracy: {rcorrect}/{len(test_DL.dataset)} ({accuracy_e:.1f} %)")
